# AG_PRAXIS NB08b — Adaptive Earliness

NB08 asked how much of a window a model needs by fixing the amount in advance: four budgets,
5, 10, 25 and 50 records, a model trained at each, every model scored on the same windows.
That answers how well a model does when it is told how long it may look. It does not answer
what happens when the model decides for itself.

The difference matters because the classes are not alike. A flood is recognisable almost at
once and a scan is not, so a budget that suits one wastes observation on the other or starves
it. A single model that stops as soon as it is confident would spend its records where they
are needed, and the price of that is a decision rule with a threshold in it.

So this notebook trains one model on prefixes of every length rather than one model per
budget, and then reads it one record at a time: at each record it looks at the top class's
softmax score, and the first time that score crosses tau it stops and answers. A window whose
score never crosses tau is answered on all fifty records.

What comes out is a curve. Every tau gives a mean number of records used and a macro-F1 at
the point the model stopped, and the two move against each other. NB08's four fixed budgets
are plotted on the same axes, because they are the reference this is measured against.

This measures earliness at a confidence trigger. It does not re-test either hypothesis. H2
stands exactly as measured on saturation under `PREREGISTRATION.md` Amendment 12. Records to
threshold remains a reported result, right-censored under Amendment 11. Neither is
recomputed here, and nothing below bears on either.

The usual first cell: Drive, the repository, and the commit this ran at.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Configuration from `config/base.yaml` and the parent's own config, with the fixed quantities
asserted rather than assumed. The parent is the NB06 sequence model, and this run keeps its
architecture, its split, its features, its seed and its ten epochs at batch 32. What changes
is the input it trains on.

In [ ]:
import gc
import json
import random
import time

import numpy as np
import pandas as pd

from baselines import mohammadi as mo
from src import inventory as inv
from src import runs as rn
from src import sequence as sq

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
BATCH_SIZE = int(CFG["training"]["batch_size"])
EPOCHS = int(CFG["training"]["epochs"])
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = ARTIFACTS / ("NB08b_fast" if FAST else "NB08b")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"], "NB04_manifest.json")
PARENT_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06",
     ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class"], "the parent run")
ARRAY_DIR = first_existing([ARTIFACTS / "NB04"], "the NB04 sequence arrays")

MANIFEST = json.loads(MANIFEST_PATH.read_text())
PARENT = json.loads((PARENT_DIR / "config.json").read_text())
PARENT_METRICS = json.loads((PARENT_DIR / "metrics.json").read_text())

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
TEST_BY_CLASS = dict(MANIFEST["arrays"]["sequences_test"]["by_class"])

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 240)

print(f"pass           : {'FAST, not a result' if FAST else 'FULL, the one the repository takes'}")
print(f"seed           : {SEED}")
print(f"window, stride : {WINDOW}, {STRIDE}")
print(f"batch, epochs  : {BATCH_SIZE}, {EPOCHS}")
print(f"features       : {len(FEATURES)}, classes {len(CLASSES)}")
print(f"parent         : {PARENT['run_id']}, macro F1 {PARENT_METRICS['macro_f1']:.4f}, "
      f"{PARENT_METRICS['n_parameters']:,} parameters")
print(f"artefacts      : {OUT_DIR}")

assert len(FEATURES) == 44 and len(CLASSES) == 19
assert (WINDOW, STRIDE) == (50, 25)
assert (BATCH_SIZE, EPOCHS) == (32, 10)
assert MANIFEST["split"]["protocol"] == "two_tier"
assert int(PARENT_METRICS["n_parameters"]) == 214227
assert int(PARENT_METRICS["n_test"]) == 49159

The fixed budgets this is measured against come from two places, not one. NB08 trained the
models at 5, 10 and 25 records and wrote them under its own artefacts. It did not train one
at 50: `PREREGISTRATION.md` Amendment 11 fixes budget 50 as the existing
`sequence_cnn_lstm_19class` run, not retrained, so that figure is the parent's own and is
read from the parent's directory.

All four have to have been scored on the same windows as this run, or the curve and the
points sit on different problems. That is asserted rather than assumed.

In [ ]:
NB08_DIR = first_existing([REPO_ROOT / "data" / "processed" / "NB08", ARTIFACTS / "NB08"],
                          "the NB08 budget runs")

FIXED = []
for k, source in ((5, NB08_DIR / "sequence_budget_05"), (10, NB08_DIR / "sequence_budget_10"),
                  (25, NB08_DIR / "sequence_budget_25"), (50, PARENT_DIR)):
    metrics = json.loads((source / "metrics.json").read_text())
    FIXED.append({"k": k, "macro_f1": float(metrics["macro_f1"]),
                  "n_test": int(metrics["n_test"]), "from": str(source)})

FIXED = pd.DataFrame(FIXED)
print("NB08's fixed budgets, the reference this run is measured against")
print(FIXED.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"k = 50 is the parent, not retrained, under Amendment 11: {PARENT['run_id']}")

assert (FIXED["n_test"] == 49159).all(), (
    "one of the fixed-budget runs was scored on a different number of windows, so its point "
    "does not belong on these axes"
)
assert len(FIXED) == 4

Seeding before anything is built, and a look at the backend.

In [ ]:
import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    named = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        named.append(str(details.get("device_name", device.name)))
    return ", ".join(named)


# Recorded into every run's config, so a run says what it ran under rather than
# leaving it to be remembered, and so a later dry run has something real to compare
# its own versions against.
ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}
print(f"seeded with : {SEED}")
print(f"tensorflow  : {tf.__version__}")
print(f"keras       : {keras.__version__}, backend {keras.backend.backend()}")
print(f"gpu         : {[d.name for d in GPUS] if GPUS else 'none, this will be slow'}")
print(f"environment : {ENVIRONMENT}")

The model is the parent's, built over an unfixed number of records rather than a fixed fifty.
The encoder reads one record at a time and the LSTM reads across however many it is handed,
so neither holds a weight that depends on the length, and the parameter count is the parent's
214,227 unchanged. That is checked here rather than asserted in prose: if this model had a
different number of parameters it would not be the parent's architecture and no comparison
below would hold.

In [ ]:
SPECIMEN = sq.build_model(len(FEATURES), len(CLASSES), window=None, lstm_units=sq.LSTM_UNITS)
ENCODER_CHECK = sq.encoder_matches_baseline(SPECIMEN, len(FEATURES), len(CLASSES))

print("the record encoder against the published network, layer by layer")
print(pd.DataFrame(ENCODER_CHECK["rows"]).to_string(index=False))
print()
print(f"input      : {SPECIMEN.input_shape}   (None in the record axis)")
print(f"parameters : {SPECIMEN.count_params():,}")

assert ENCODER_CHECK["agrees"]
assert SPECIMEN.input_shape == (None, None, len(FEATURES), 1)
assert int(SPECIMEN.count_params()) == int(PARENT_METRICS["n_parameters"]), (
    "an unfixed record axis changed the parameter count, which it should not"
)
del SPECIMEN
gc.collect()

The windows are NB04's, unchanged, with the same checks NB07 and NB07b run on them. The test
partition has to be the 49,159 windows every run in this project is scored on.

In [ ]:
def read_partition(name):
    path = ARRAY_DIR / f"sequences_{name}.npz"
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if [str(v) for v in npz["classes"]] != CLASSES:
            raise ValueError(f"{path.name} holds different classes than the manifest lists")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X, y = npz["X"], npz["y"].astype("int64")
    print(f"  {path.name:<24} {str(X.shape):>22}   {X.dtype}")
    assert X.shape[1:] == (WINDOW, len(FEATURES))
    assert np.isfinite(X).all()
    return X, y


print("reading the sequence arrays")
X_TRAIN_RAW, Y_TRAIN = read_partition("train")
X_TEST_RAW, Y_TEST = read_partition("test")
X_TRAIN = sq.reshape(X_TRAIN_RAW)
X_TEST = sq.reshape(X_TEST_RAW)
del X_TRAIN_RAW, X_TEST_RAW
gc.collect()

print()
print(f"train windows : {len(Y_TRAIN):,}")
print(f"test windows  : {len(Y_TEST):,}")
assert len(Y_TEST) == 49159, f"the test partition holds {len(Y_TEST):,} windows, not 49,159"

The one change from the parent is what the model is trained on. Instead of every window at
fifty records, each batch is cut to a single length drawn uniformly from one to fifty, so
over ten epochs the model is trained at every length it can later be asked to stop at.

The length is drawn per batch rather than per window because an array has one shape. Drawing
uniformly gives a five-record prefix the same share of the training as a fifty-record one,
although it carries a fraction of the information; that is a property of the design rather
than an oversight, and `DECISIONS.md` records it along with the two alternatives that were
considered and rejected. No length weighting is applied, because that would be a second
choice inside a run whose one change is the training input.

In [ ]:
RUN_CONFIG = dict(PARENT)
RUN_CONFIG["run_id"] = "adaptive_prefix_19class"
RUN_CONFIG["parent"] = PARENT["run_id"]
RUN_CONFIG["training_input"] = "prefixes of length drawn uniformly from 1 to 50, one per batch"
RUN_CONFIG["observed"] = dict(PARENT.get("observed", {}))
RUN_CONFIG["observed"].update({
    "environment": ENVIRONMENT,
    "run_date": RUN_DATE,
    "mode": "fast" if FAST else "full",
    "changed_from_parent": ["training_input"],
    "input": "a window of 50 records at stride 25, cut to a length drawn per batch",
    "input_shape": [None, len(FEATURES), 1],
    "registered_under": "PREREGISTRATION.md Amendment 15",
    "scored_at": "the full 50 records; the halting rule is applied afterwards",
})

print("one change from the parent:", sorted(rn.assert_single_change(RUN_CONFIG, PARENT)))
print()

RUN = sq.fit_and_save(
    OUT_DIR,
    RUN_CONFIG["run_id"],
    X_train=X_TRAIN,
    y_train=Y_TRAIN,
    X_test=X_TEST,
    y_test=Y_TEST,
    classes=CLASSES,
    config=RUN_CONFIG,
    parent=PARENT,
    window=None,
    n_features=len(FEATURES),
    lstm_units=sq.LSTM_UNITS,
    seed=SEED,
    checkpoint=True,
    verbose=2,
    mixed_length=True,
)

In [ ]:
M = RUN["metrics"]
print(f"trained on {M['n_train']:,} windows, scored on {M['n_test']:,}")
print(f"macro F1 at the full 50 records : {M['macro_f1']:.4f}")
print(f"parent, trained and scored at 50: {PARENT_METRICS['macro_f1']:.4f}")
print(f"parameters                      : {M['n_parameters']:,}")
print(f"trained in                      : {M['train_seconds']:,.0f}s")
print()
lengths = M["prefix_lengths"]
print(f"prefix lengths drawn: {lengths['n_batches']:,} batches, {lengths['min']} to "
      f"{lengths['max']}, mean {lengths['mean']:.2f}")
print("The row above is the run scored the way every other run is scored, at the full window.")
print("It is not the adaptive result. The halting rule is applied below.")

Now the halting rule. Each window is stepped through one record at a time, and the first time
the top class's softmax score crosses tau the model stops and answers with that class. A
window whose score never crosses tau is answered on all fifty.

Rather than stepping window by window, the model is asked for its answer at every prefix
length once, which gives a score and a class for all fifty positions of all 49,159 windows.
The halting rule is then read off that table, and the same table serves all twelve values of
tau. Stepping each window separately would compute exactly the same numbers many times over.

In [ ]:
PREDICT_BATCH = 512

# fit_and_save returns where the model was written rather than the model, so the run is
# read back from its own artefact. That is also a check that the file loads.
RUN_MODEL = keras.models.load_model(RUN["model_file"])
print(f"loaded {RUN['model_file']}")
print(f"input {RUN_MODEL.input_shape}, parameters {RUN_MODEL.count_params():,}")
assert int(RUN_MODEL.count_params()) == int(M["n_parameters"])

started = time.time()
TOP_SCORE = np.empty((len(Y_TEST), WINDOW), dtype="float32")
TOP_CLASS = np.empty((len(Y_TEST), WINDOW), dtype="int16")

for k in range(1, WINDOW + 1):
    probabilities = RUN_MODEL.predict(
        np.ascontiguousarray(X_TEST[:, :k]), batch_size=PREDICT_BATCH, verbose=0
    )
    TOP_SCORE[:, k - 1] = probabilities.max(axis=1)
    TOP_CLASS[:, k - 1] = probabilities.argmax(axis=1)
    del probabilities
    if k % 10 == 0 or k == WINDOW:
        print(f"  scored at {k:>2} records, {time.time() - started:.0f}s")

print()
print(f"score and class at every prefix length: {TOP_SCORE.shape}")
print(f"took {time.time() - started:.0f}s")
print(f"mean top score at 1 record : {TOP_SCORE[:, 0].mean():.4f}")
print(f"mean top score at 50       : {TOP_SCORE[:, -1].mean():.4f}")

The tau grid is fixed in `PREREGISTRATION.md` Amendment 15: 0.50 to 0.90 in steps of 0.05,
then 0.95, 0.97 and 0.99. Twelve values. The grid is not uniform, and the amendment records
why: the resolution is concentrated in the upper tail, because that is where windows begin
failing to reach tau within fifty records, and the share of windows that never trigger is one
of the quantities being reported.

In [ ]:
TAUS = [round(0.50 + 0.05 * i, 2) for i in range(9)] + [0.95, 0.97, 0.99]

print(f"tau grid: {TAUS}")
print(f"values  : {len(TAUS)}")
print(f"steps   : {[round(TAUS[i + 1] - TAUS[i], 2) for i in range(len(TAUS) - 1)]}")

assert len(TAUS) == 12
assert TAUS[:9] == [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
assert TAUS[9:] == [0.95, 0.97, 0.99]

For each tau: where each window halted, what it answered there, the macro-F1 of those
answers, the mean number of records used, and the share of windows that never reached tau.

Earliness is averaged over correctly classified windows only, as Amendment 15 fixes. A window
the model got wrong has a number of records attached to it, but that number is how long it
took to be wrong, which is not what earliness is meant to report.

In [ ]:
from sklearn.metrics import f1_score

LABELS = list(range(len(CLASSES)))


def halt_at(tau):
    """Where each window crosses tau, what it answers there, and whether it ever crossed."""
    crossed = TOP_SCORE >= float(tau)
    ever = crossed.any(axis=1)
    position = np.where(ever, crossed.argmax(axis=1), WINDOW - 1)
    answer = TOP_CLASS[np.arange(len(position)), position].astype("int64")
    return position + 1, answer, ever


rows, per_class = [], []
for tau in TAUS:
    records, answer, ever = halt_at(tau)
    correct = answer == Y_TEST
    macro = float(f1_score(Y_TEST, answer, labels=LABELS, average="macro", zero_division=0))
    weighted = float(f1_score(Y_TEST, answer, labels=LABELS, average="weighted", zero_division=0))
    class_f1 = f1_score(Y_TEST, answer, labels=LABELS, average=None, zero_division=0)

    rows.append({
        "tau": tau,
        "mean earliness": float(records[correct].mean()) if correct.any() else float("nan"),
        "macro F1": macro,
        "weighted F1": weighted,
        "accuracy": float(correct.mean()),
        "never reached tau": float((~ever).mean()),
        "mean records, all windows": float(records.mean()),
    })
    for code_, name in enumerate(CLASSES):
        of_class = Y_TEST == code_
        right = of_class & correct
        per_class.append({
            "tau": tau, "class": name,
            "earliness": float(records[right].mean()) if right.any() else float("nan"),
            "f1": float(class_f1[code_]),
            "never reached tau": float((~ever[of_class]).mean()),
            "test windows": int(of_class.sum()),
            "n_correct": int(right.sum()),
        })

CURVE = pd.DataFrame(rows)
PER_CLASS = pd.DataFrame(per_class)

print("one row per tau, over all 49,159 test windows")
print(CURVE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("Earliness is the mean over correctly classified windows only. The last column is the")
print("mean over every window, which is not the reported quantity and is here so the two can")
print("be told apart.")

In [ ]:
print("per-class earliness, one column per tau, over that class's correct windows")
earliness_table = PER_CLASS.pivot(index="class", columns="tau", values="earliness")
print(earliness_table.to_string(float_format=lambda v: f"{v:.1f}"))
print()
print("per-class F1 at the halt point")
f1_table = PER_CLASS.pivot(index="class", columns="tau", values="f1")
print(f1_table.to_string(float_format=lambda v: f"{v:.4f}"))
print()
print("share of each class's windows that never reached tau")
never_table = PER_CLASS.pivot(index="class", columns="tau", values="never reached tau")
print(never_table.to_string(float_format=lambda v: f"{v:.3f}"))

The figure. Earliness on one axis and macro-F1 on the other, the twelve taus joined in order,
and NB08's four fixed budgets on the same axes as points. The fixed budgets are placed at
their own budget in records, since a model told to use five records uses five.

The figure is written under a descriptive name and carries no figure number. Numbering
belongs to the chapter it lands in, and a number chosen here would be wrong as soon as a
figure is added ahead of it.

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID, RULE = "#0b0b0b", "#52514e", "#e6e5e1", "#c9c8c3"
LINE, POINT = "#3f6fb0", "#d1622b"

fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(CURVE["mean earliness"], CURVE["macro F1"], "-o", color=LINE, markersize=5,
        linewidth=1.4, label="adaptive halting, one model, tau 0.50 to 0.99")
for tau, x, y in zip(CURVE["tau"], CURVE["mean earliness"], CURVE["macro F1"]):
    ax.annotate(f"{tau:.2f}", (x, y), textcoords="offset points",
                xytext=(0, 7), ha="center", fontsize=7, color=MUTED)
ax.plot(FIXED["k"], FIXED["macro_f1"], "s", color=POINT, markersize=7,
        label="NB08 fixed budgets, a model trained at each")
for k, y in zip(FIXED["k"], FIXED["macro_f1"]):
    ax.annotate(f"k = {int(k)}", (k, y), textcoords="offset points",
                xytext=(0, -14), ha="center", fontsize=8, color=POINT)

ax.set_xlabel("records observed, mean over correctly classified windows", color=MUTED, fontsize=9)
ax.set_ylabel("macro-F1 at the point the model answered", color=MUTED, fontsize=9)
ax.set_title(
    "Earliness against macro-F1 under a per-class confidence trigger,\nagainst the four "
    "fixed observation budgets",
    color=INK, fontsize=11, loc="left", pad=12,
)
ax.grid(color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("bottom", "left"):
    ax.spines[side].set_color(RULE)
ax.tick_params(colors=MUTED, length=0)
ax.legend(frameon=False, fontsize=8, loc="lower right")

FIGURE_PATH = OUT_DIR / "NB08b_earliness_curve.png"
fig.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight", facecolor="white")
print(f"wrote {FIGURE_PATH}")
plt.show()
plt.close(fig)

Everything the sweep produced, written beside the run so the table can be read back without
this notebook.

In [ ]:
DOCUMENT = {
    "generated_by": "AG_PRAXIS_NB08b_adaptive_earliness.ipynb",
    "generated_on": RUN_DATE,
    "git_sha": GIT_SHA,
    "git_dirty": GIT_DIRTY,
    "seed": SEED,
    "is_fast_pass": bool(FAST),
    "registered_under": "PREREGISTRATION.md Amendment 15",
    "run": RUN_CONFIG["run_id"],
    "parent": PARENT["run_id"],
    "training_input": RUN_CONFIG["training_input"],
    "halting_rule": (
        "step through the window one record at a time, halt at the first record where the "
        "top class's softmax score crosses tau and answer there; a window that never crosses "
        "tau is answered on all 50 records"
    ),
    "earliness": "mean records over correctly classified windows only",
    "tau_grid": TAUS,
    "n_test": int(len(Y_TEST)),
    "curve": CURVE.to_dict(orient="records"),
    "per_class": PER_CLASS.to_dict(orient="records"),
    "fixed_budgets": FIXED.to_dict(orient="records"),
    "does_not_retest": (
        "H2 stands as measured on saturation under Amendment 12, and records to threshold "
        "remains a reported result right-censored under Amendment 11. Neither is recomputed "
        "here."
    ),
    "figure": FIGURE_PATH.name,
}
SUMMARY_PATH = OUT_DIR / "adaptive_earliness.json"
SUMMARY_PATH.write_text(json.dumps(DOCUMENT, indent=2, default=str) + "\n")
print(f"wrote {SUMMARY_PATH}")
print(f"the run itself is in {RUN['run_dir']}")

The ledger entry, ready to paste into `RESULTS_LEDGER.md`.

In [ ]:
if FAST:
    status = "DO NOT ENTER, fast pass"
elif GIT_DIRTY:
    status = "reported result, working tree dirty"
else:
    status = "reported result, not a hypothesis test"

best = CURVE.loc[CURVE["macro F1"].idxmax()]
ledger = f"""
### NB08b — adaptive earliness ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB08b_adaptive_earliness.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| status | {status} |
| registered under | PREREGISTRATION.md Amendment 15 |
| parent | {PARENT['run_id']}, one change: training_input |
| training input | prefixes of length drawn uniformly from 1 to 50, one length per batch |
| model | one model over an unfixed record axis, {M['n_parameters']:,} parameters, the parent's |
| test items | {M['n_test']:,} windows, the same as NB08's budget runs and the parent |
| macro F1 at the full 50 records | {M['macro_f1']:.4f}, against the parent's {PARENT_METRICS['macro_f1']:.4f} |
| halting rule | first record where the top class's softmax score crosses tau; a window that never crosses is answered at 50 |
| tau grid | {", ".join(f"{t:g}" for t in TAUS)} |
| earliness | mean records over correctly classified windows only |
| highest macro F1 on the sweep | {best['macro F1']:.4f} at tau {best['tau']:g}, mean earliness {best['mean earliness']:.2f} records, {best['never reached tau']:.3f} of windows never reached tau |
| at tau 0.50 | macro F1 {CURVE.iloc[0]['macro F1']:.4f}, earliness {CURVE.iloc[0]['mean earliness']:.2f}, never reached {CURVE.iloc[0]['never reached tau']:.3f} |
| at tau 0.99 | macro F1 {CURVE.iloc[-1]['macro F1']:.4f}, earliness {CURVE.iloc[-1]['mean earliness']:.2f}, never reached {CURVE.iloc[-1]['never reached tau']:.3f} |
| fixed budgets, for reference | {" · ".join(f"k={int(r.k)} {r.macro_f1:.4f}" for r in FIXED.itertuples())} |
| seed replicates | none. Single run at seed {SEED} |
| not computed | significance. That is NB08's |
| artefacts | {OUT_DIR}, holding adaptive_earliness.json, the figure and the run |

H2 stands as measured on saturation under Amendment 12. Records to threshold remains a
reported result, right-censored under Amendment 11. This notebook re-tests neither.
"""

print(ledger)